In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 21. RLHF — 인간 피드백 강화학습 ⭐ [CPU/GPU 벤치마크 ⑨]

> **학습 목표**
> - 강화학습 기본 개념(MDP, 정책, 보상, 가치)을 LLM에 매핑한다
> - 정책 경사(Policy Gradient)와 REINFORCE 알고리즘을 유도한다
> - PPO의 clipped objective를 이해하고 구현한다
> - 보상 모델(RM) 학습을 설명한다

## 21.1 SFT의 한계와 정렬 문제

SFT 모델은 "응답 형식"은 배웠지만:
- 인간이 **선호하는** 응답인지는 모름
- 유해하거나 부정확해도 그럴듯한 답을 생성 가능
- "정렬(alignment)" 필요: 모델을 인간 가치에 맞춤

## 21.2 RLHF 3단계

1. **SFT**: 지시 따르기 (Ch 20)
2. **Reward Model (RM)**: 응답 쌍에 점수 매기는 모델 학습
3. **PPO**: RM을 보상으로 정책(=LLM) 강화학습

## 21.3 강화학습 기초 — MDP

**MDP** (Markov Decision Process): $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$
- $\mathcal{S}$: 상태 공간
- $\mathcal{A}$: 행동 공간
- $P(s'|s, a)$: 전이 확률
- $R(s, a)$: 보상
- $\gamma$: 할인율

LLM 매핑:
- 상태 $s_t$ = 지금까지 생성한 토큰 $[w_0, \ldots, w_{t-1}]$
- 행동 $a_t$ = 다음 토큰 $w_t$
- 정책 $\pi_\theta(a_t | s_t) = P(w_t | w_{<t}; \theta)$
- 보상 $R$ = RM이 매긴 응답 점수 (보통 마지막 토큰에서만)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 어휘 크기 (toy)
vocab_size = 100  # 이 챕터 전체에서 사용
print(f"vocab_size (toy): {vocab_size}")

## 21.4 정책 경사 정리 (REINFORCE)

목표: $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$ 최대화.

$$\nabla J(\theta) = \mathbb{E}_{\tau} \left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

- $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$: 할인된 누적 보상
- $\nabla \log \pi$: 로그 확률의 그래디언트 → "이 행동을 더 할 가능성" 조절
- $G_t$로 가중 → 좋은 보상이면 그 행동 확률 증가

**Advantage** $A_t = G_t - V(s_t)$: 베이스라인 $V$ 빼서 분산 감소.


In [ ]:
# REINFORCE 예시 (간단한 toy MDP)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 환경: 간단한 bandit (3개 팔, 보상 분포 다름)
true_rewards = [0.2, 0.5, 0.8]  # 팔 2가 최고

class PolicyNet(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(1, 16)
        self.head = nn.Linear(16, n_actions)
    def forward(self, x):
        return self.head(F.relu(self.fc(x)))

policy = PolicyNet(3)
opt = torch.optim.Adam(policy.parameters(), lr=0.01)

n_episodes = 500
all_rewards = []
for ep in range(n_episodes):
    # 에피소드: 1 step (bandit)
    state = torch.tensor([[1.0]])
    logits = policy(state)
    probs = F.softmax(logits, dim=-1)
    action = torch.multinomial(probs, 1).item()
    reward = true_rewards[action] + np.random.randn() * 0.05

    # REINFORCE: log_prob * reward
    log_prob = F.log_softmax(logits, dim=-1)[0, action]
    loss = -log_prob * reward  # 최대화 → 음의 최소화
    opt.zero_grad()
    loss.backward()
    opt.step()
    all_rewards.append(reward)

print(f"초기 50 에피소드 Mean Reward: {np.mean(all_rewards[:50]):.3f}")
print(f"마지막 50 에피소드 Mean Reward: {np.mean(all_rewards[-50:]):.3f}")
print(f"Theory Maximum: {max(true_rewards):.3f}")


## 21.5 PPO — Trust Region과 Clipped Objective

REINFORCE는 분산 큼. PPO는 정책이 한 번에 너무 바뀌지 않게 제한.

**비율**: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$

**Clipped Objective**:
$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[\min\left(r_t \hat{A}_t, \, \mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

- $\epsilon \approx 0.2$: 비율의 허용 범위
- min으로 clip → 정책이 너무 크게 바뀌는 것 방지
- trust region의 근사


In [ ]:
# PPO 예시 (toy 문제)
import torch
import torch.nn as nn
import torch.nn.functional as F

class Policy(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(4, 32)
        self.head = nn.Linear(32, n_actions)
    def forward(self, x):
        return self.head(F.tanh(self.fc(x)))

# 간단한 환경 (랜덤)
def env_step(state, action):
    reward = float((state.sum() + action - 2) / 10)  # 간단한 보상
    next_state = state + torch.randn(4) * 0.1
    return next_state, reward, False

policy = Policy(3)
opt = torch.optim.Adam(policy.parameters(), lr=3e-4)

# PPO 업데이트
def ppo_update(policy, opt, states, actions, old_log_probs, advantages, returns, values,
               epsilon=0.2, n_epochs=4):
    for _ in range(n_epochs):
        logits = policy(states)
        new_log_probs = F.log_softmax(logits, dim=-1)
        new_log_probs_a = new_log_probs.gather(1, actions.unsqueeze(1)).squeeze()

        ratio = torch.exp(new_log_probs_a - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()

        # 가치 함수 손실 (간단)
        critic_loss = F.mse_loss(values.squeeze(), returns)

        loss = actor_loss + 0.5 * critic_loss
        opt.zero_grad()
        loss.backward()
        opt.step()

# 학습 시뮬레이션
n_iterations = 50
all_rewards = []
for it in range(n_iterations):
    # rollout 수집
    states, actions, rewards, old_log_probs = [], [], [], []
    state = torch.randn(1, 4)
    ep_reward = 0
    for step in range(20):
        with torch.no_grad():
            logits = policy(state)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            old_log_prob = dist.log_prob(action)
        next_state, reward, done = env_step(state.squeeze(), action.item())
        states.append(state.squeeze())
        actions.append(action)
        rewards.append(reward)
        old_log_probs.append(old_log_prob)
        ep_reward += reward
        state = next_state.unsqueeze(0)
    all_rewards.append(ep_reward / 20)

    # advantage 계산 (간단: 보상 그대로)
    advantages = torch.tensor(rewards)
    returns = torch.tensor(rewards)
    values = torch.zeros_like(returns)  # 베이스라인 없음 (간소화)

    ppo_update(policy, opt, torch.stack(states), torch.tensor(actions),
               torch.stack(old_log_probs), advantages, returns, values)

print(f"초기 10 에피소드 Mean Reward: {np.mean(all_rewards[:10]):.4f}")
print(f"마지막 10 에피소드 Mean Reward: {np.mean(all_rewards[-10:]):.4f}")


## 21.6 보상 모델 (RM) 학습

RM은 응답에 점수를 매기는 모델. **선호 데이터** (chosen vs rejected)로 학습.

**Bradley-Terry 모델**:
$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

손실 (NLL):
$$\mathcal{L}_{\text{RM}} = -\log \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

- $\mathbf{y}_w$: 선호(chosen) 응답
- $\mathbf{y}_l$: 기각(rejected) 응답
- $r(\mathbf{x}, \mathbf{y})$: RM이 매긴 점수

RM 구조: SFT 모델에 스칼라 출력층 추가.


In [ ]:
# RM 학습 시뮬레이션
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.LSTM(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, 1)  # 스칼라 보상

    def forward(self, x):
        emb = self.emb(x)
        _, (h, _) = self.rnn(emb)
        return self.head(h.squeeze(0))  # (B,)

# toy 선호 데이터
n_samples = 100
torch.manual_seed(0)
rm = RewardModel(vocab_size=vocab_size, d_model=32)
opt = torch.optim.AdamW(rm.parameters(), lr=1e-3)

# 랜덤 chosen/rejected pairs (시뮬레이션)
print("RM Training 시뮬레이션:")
losses = []
for step in range(200):
    # 가짜 데이터: 짧은 시퀀스
    chosen = torch.randint(0, vocab_size, (8, 16))
    rejected = torch.randint(0, vocab_size, (8, 16))
    # 인위적 차이: chosen의 평균 ID가 더 작다고 가정
    # (실제로는 RM이 학습할 패턴)
    r_chosen = rm(chosen).squeeze(-1)
    r_rejected = rm(rejected).squeeze(-1)
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('RM Loss')
plt.title('Reward Model Training (Bradley-Terry)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch21_rm_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"최종 RM loss: {losses[-1]:.4f} (Theory Minimum ≈ 0.69 = -log(0.5) 랜덤)")


## 21.7 KL 패널티 — 참조 모델에서 너무 멀어지지 않도록

RLHF에서 RM 점수만 최대화하면 모델이 이상한 텍스트를 생성할 수 있음 (reward hacking).

해결: 참조 모델(SFT 모델)에서 너무 멀어지지 않게 KL 패널티.

$$R_{\text{total}} = R_{\text{RM}}(\mathbf{x}, \mathbf{y}) - \beta \, D_{\text{KL}}(\pi_\theta(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

- $\beta$: KL 강도 (보통 0.01~0.5)
- $\pi_{\text{ref}}$: SFT 모델 (고정)
- 토큰별 KL: $\sum_t \log \frac{\pi_\theta(w_t|...)}{\pi_{\text{ref}}(w_t|...)}$


In [ ]:
# KL 발산 계산 (정책 vs 참조)
def token_kl_div(policy_logits, ref_logits):
    """토큰별 KL Divergence."""
    p = F.softmax(policy_logits, dim=-1)
    log_p = F.log_softmax(policy_logits, dim=-1)
    log_q = F.log_softmax(ref_logits, dim=-1)
    return (p * (log_p - log_q)).sum(dim=-1)  # (B, T)

# 시뮬레이션
torch.manual_seed(0)
B, T, V = 4, 16, 100
ref_logits = torch.randn(B, T, V)
policy_logits = ref_logits + torch.randn(B, T, V) * 0.5  # 약간 다른 Policy

kl = token_kl_div(policy_logits, ref_logits)
print(f"KL 발산 shape: {kl.shape}")
print(f"Mean KL: {kl.mean():.4f}")
print(f"Maximum KL: {kl.max():.4f}")
print("\n=> Policy이 참조에서 멀어질수록 KL Increase. RLHF는 이를 패널티로 사용.")


## 21.8 [CPU/GPU 벤치마크 ⑨] SFT vs RLHF 학습 비교

RLHF는 모델 4개 필요: 정책, 참조, RM, 가치함수. 메모리·시간 3~4배.


In [ ]:
# RLHF 메모리 분석 (시뮬레이션)
print("RLHF Training에 필요한 Model:")
print("  1. Policy model (Training됨)")
print("  2. Reference model (고정, KL용)")
print("  3. Reward model (고정)")
print("  4. Value/Critic model (Training됨, optional)")
print()

# 예시: 7B LLaMA RLHF
model_size_gb = 7 * 2 * 4  # 7B params * 2 bytes (FP16) * 4 models? No:
# 실제로: policy (FP32 grad, FP32 opt state)
# AdamW: param + grad + m + v = 4배
# 7B 모델 FP16: 14GB
# AdamW 상태 FP32: 7B * 8 bytes = 56GB
# policy: 14 + 56 = 70GB
# reference: 14GB
# RM: 14GB
# value: 14 + 56 = 70GB
# 총: 약 168GB → A100 80GB 2개 필요

print("7B Model RLHF Memory Estimation:")
print(f"  Policy (FP16 + AdamW FP32): ~70GB")
print(f"  Reference (FP16, 고정): ~14GB")
print(f"  Reward Model (FP16, 고정): ~14GB")
print(f"  Value (FP16 + AdamW): ~70GB")
print(f"  총: ~168GB (A100 80GB × 2 필요)")
print("\n=> RLHF는 매우 비싸다. DPO(Ch 22)가 더 Efficiency적인 대안.")


## 21.9 RLHF의 함정

- **보상 해킹(reward hacking)**: RM 점수는 높지만 실제로는 나쁜 응답
- **모드 붕괴(mode collapse)**: 다양성 잃고 하나의 패턴만 생성
- **불안정성**: PPO 하이퍼파라미터 민감

## 21.10 핵심 정리

| 개념 | 수식 | 의미 |
|---|---|---|
| 정책 경사 | $\nabla J = \mathbb{E}[\nabla\log\pi \cdot G]$ | REINFORCE |
| Advantage | $A = G - V$ | 베이스라인 제거 |
| PPO clip | $\min(rA, \text{clip}(r)A)$ | trust region |
| RM 손실 | $-\log\sigma(r_w - r_l)$ | Bradley-Terry |
| KL 패널티 | $R - \beta D_{KL}$ | 참조 유지 |

## 연습문제

1. REINFORCE로 CartPole 같은 toy 환경을 학습하라.
2. PPO에서 $\epsilon = 0.1, 0.2, 0.3$을 비교하고 영향을 분석하라.
3. RM 학습에서 chosen과 rejected가 비슷할 때 vs 다를 때 loss를 비교하라.
4. KL 패널티 $\beta = 0, 0.1, 1.0$에서 정책 변화를 시뮬레이션하라.
5. RLHF 대비 DPO의 장점을 메모리 관점에서 설명하라.

> 해설: `solutions/ch21_solutions.ipynb`
